# 7. Regional Bathymetry Workflows

This notebook demonstrates the two regional bathymetry pipelines in `mom6_forge`:

| Pipeline | Method | Best for |
|---|---|---|
| **A** | `direct_xesmf_regrid` | Fine grids (< 5 km), fast runs |
| **B** | `high_res_regrid` | Coarser grids (≳ 0.05°), topo drag |

Both pipelines support an external land/ocean mask. Two mask generation methods are available:
- `generate_mask_ocean_frac` — Monte-Carlo ocean fraction from raw GEBCO
- `generate_mask_cartopy` — Cartopy Natural Earth coastline rasterisation

> **Note:** This notebook documents the intended API for methods currently under development on the `global_workflow_convert` branch. Cells marked `[PROPOSED API]` will raise `AttributeError` until the implementation is merged.

## Setup

In [ ]:
from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np

Create a test grid over the Gulf of Mexico at 0.25° resolution.
This resolution (~25 km) is coarse enough that both mask quality and Cressman interpolation matter.

In [ ]:
grid = Grid(
    resolution=0.25,
    xstart=260.0,
    lenx=30.0,
    ystart=18.0,
    leny=12.0,
    name="gulf_of_mexico",
)

topo = Topo(grid, min_depth=5.0, version_control_dir="TopoLibrary")

GEBCO_PATH = "/path/to/gebco_2023.nc"  # update to your local GEBCO file

---
## 1. Mask Generation

Both pipelines can take an externally computed mask. The mask determines which cells are ocean — everything downstream (depth interpolation, tidal cleanup, min-depth enforcement) only applies to ocean cells.

### Option A — Ocean Fraction Mask

`generate_mask_ocean_frac` computes the fraction of each model cell that is ocean by sub-sampling the raw GEBCO data.

For each model cell:
1. `nx_sub × ny_sub` interior points are placed using bilinear interpolation of the 4 supergrid corner coordinates.
2. Each point is snapped to the nearest GEBCO pixel.
3. `OCN_FRAC` = fraction of sub-points with depth > `mask_hmin`.
4. Cells with `OCN_FRAC ≥ mask_threshold` become ocean.

This method also stores per-cell depth statistics (`D_mean`, `D_min`, `D_max`, `D2_mean`) needed later for topo drag.

In [ ]:
# [PROPOSED API]
mask_ocn_frac, ocn_frac = topo.generate_mask_ocean_frac(
    bathymetry_path=GEBCO_PATH,
    nx_sub=5,           # 5x5 = 25 sub-points per cell
    ny_sub=5,
    mask_threshold=0.5, # >50% ocean -> ocean cell
    mask_hmin=0.0,      # sub-points with depth > 0 count as ocean
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ocn_frac.plot(ax=axes[0], cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Ocean Fraction (OCN_FRAC)")
mask_ocn_frac.plot(ax=axes[1], cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Binary Mask (threshold=0.5)")
plt.tight_layout()

### Option B — Cartopy Coastline Mask

`generate_mask_cartopy` rasterises Natural Earth coastline vectors onto the model grid. It is faster and does not depend on the bathymetry dataset, but the mask is not derived from the depth data so land/ocean boundaries may not align exactly with GEBCO.

In [ ]:
# [PROPOSED API]
mask_cartopy = topo.generate_mask_cartopy(resolution="50m")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mask_ocn_frac.plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Ocean Fraction Mask")
mask_cartopy.plot(ax=axes[1], cmap="Blues")
axes[1].set_title("Cartopy Mask")
plt.suptitle("Mask comparison at coastlines")
plt.tight_layout()

---
## 2. Pipeline A — `direct_xesmf_regrid`

The standard pipeline. Regrids GEBCO with `xesmf` (bilinear or conservative), then runs `tidy_dataset` for lake removal, 1-cell channel cleanup, and minimum depth enforcement.

The optional `mask` argument lets you supply a pre-computed mask. Without it, the mask is derived from whether the regridded depth is positive — the original behaviour.

In [ ]:
# [PROPOSED API] — with external ocean fraction mask
topo.direct_xesmf_regrid(
    bathymetry_path=GEBCO_PATH,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    mask=mask_ocn_frac,        # omit this to use the original behaviour
    regridding_method="conservative",
)

topo.depth.plot(cmap="Blues_r")
plt.title("Pipeline A depth — xesmf + ocean fraction mask")

---
## 3. Pipeline B — `high_res_regrid`

The high-accuracy pipeline for coarser grids. Internally runs:
1. `generate_mask_ocean_frac` — builds the mask and computes depth statistics
2. `cressman_interp` — assigns ocean-mask-aware smoothed depths from raw GEBCO
3. `tidy_dataset` — lake removal, channel cleanup, min-depth enforcement

### Why Cressman instead of xesmf?

Standard regridding blends land elevations into ocean cells that straddle the coastline in the source data. Cressman weights nearby GEBCO points by distance and **excludes land source points entirely**, so coastal depth estimates are not contaminated by land elevations.

The weight function is:
$$w = \left(\frac{L^2 - r^2}{L^2 + r^2}\right)^c$$

where $r$ is the great-circle distance from the cell centre to each GEBCO point, $L = \text{smooth\_scl} \times \sqrt{A_{\text{cell}}}$, and $c$ is `cressman_exp`.

In [ ]:
# [PROPOSED API]
topo_hires = Topo(grid, min_depth=5.0, version_control_dir="TopoLibrary_hires")

topo_hires.high_res_regrid(
    bathymetry_path=GEBCO_PATH,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    nx_sub=5,
    ny_sub=5,
    mask_threshold=0.5,
    smooth_scl=2.0,       # smoothing radius = 2x local grid spacing
    cressman_exp=2.0,     # weight falloff sharpness
    hmin=5.0,             # minimum depth — should match vgrid top layer
)

topo_hires.depth.plot(cmap="Blues_r")
plt.title("Pipeline B depth — Cressman interpolation")

### Comparing the two pipelines

The largest differences appear near coastlines and shallow shelves. Interior deep ocean cells are nearly identical.

In [ ]:
# [PROPOSED API]
diff = topo_hires.depth - topo.depth

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
topo.depth.plot(ax=axes[0], cmap="Blues_r")
axes[0].set_title("Pipeline A (xesmf)")
topo_hires.depth.plot(ax=axes[1], cmap="Blues_r")
axes[1].set_title("Pipeline B (Cressman)")
diff.plot(ax=axes[2], cmap="RdBu_r", robust=True)
axes[2].set_title("Difference (B − A)")
plt.tight_layout()

---
## 4. Topo Drag Statistics

MOM6 Lee-wave and bottom-drag parameterisations need a measure of subgrid topographic roughness. This is the variance of ocean depth within each model cell:

$$h_2 = \overline{D^2} - \bar{D}^2$$

where $\overline{D^2}$ is `D2_mean` and $\bar{D}$ is `D_mean` from the Monte-Carlo sub-sampling step.

`write_topo_drag` writes a netCDF file with `h2` that MOM6 reads at initialisation when topo drag is enabled. It requires `generate_mask_ocean_frac` (or `high_res_regrid`) to have been called first.

In [ ]:
# [PROPOSED API]
topo_hires.write_topo_drag("topo_drag.nc")

drag = xr.open_dataset("topo_drag.nc")
drag.h2.plot(cmap="YlOrRd")
plt.title("Subgrid topographic variance h2 (m²)")

In [ ]:
# Verify the formula directly
# [PROPOSED API]
h2_manual = topo_hires.d2_mean - topo_hires.d_mean ** 2
print("h2 is non-negative everywhere:", bool((drag.h2 >= 0).all()))
print("Max h2:", float(drag.h2.max()), "m²")
print("Matches manual computation:", np.allclose(drag.h2.values, h2_manual.values))

---
## 5. Which pipeline to use?

`direct_xesmf_regrid` works well when your model grid resolution is close to the source dataset resolution (GEBCO is ~500 m / 15 arcseconds). As the ratio of model grid spacing to source resolution grows — i.e. each model cell covers many GEBCO pixels — `xesmf` regridding averages over increasing subgrid variability and coastal cells become more susceptible to land contamination. In that regime `high_res_regrid` with Cressman interpolation gives more accurate depths and a physically consistent mask.

For the mask, `generate_mask_ocean_frac` is the natural choice when Cressman is involved since both draw from the same GEBCO source, keeping the mask and depth field self-consistent. `generate_mask_cartopy` is a reasonable alternative for `direct_xesmf_regrid` on clean open-ocean domains where speed matters more than coastline precision.